In [2]:
import pandas as pd
import numpy as np

orders   = pd.read_csv("https://cdn.enqurious.com/documents/660a051c-25ef-4fd2-96bc-7179565c8d6f_exorders.csv")
returns  = pd.read_csv("https://cdn.enqurious.com/documents/e4961106-208d-4b54-bb13-f9dca0cddf7f_exreturns.csv")
products = pd.read_csv("https://cdn.enqurious.com/documents/bf35fdf1-d9a0-4432-9b55-6a8cb2720716_exproducts.csv")
transactions = pd.read_csv("https://cdn.enqurious.com/documents/d2628fca-f225-4482-80ee-889fdf3f95aa_extransactions.csv")

In [3]:
trans2 = transactions[['Order_ID','Product_ID']].drop_duplicates()

In [4]:
trans2.head()

,Order_ID,Product_ID
0,CA-2014-145317,FUR-BO-10001798
1,CA-2016-118689,FUR-CH-10000454
4,CA-2017-140151,OFF-LA-10000240
5,CA-2017-127180,FUR-TA-10000577
6,CA-2017-166709,OFF-ST-10000760


In [5]:
orders.head(2)

,order_id,customer_id,partner_id,ship_mode,order_status,order_purchase_date,order_approved_at,order_dispatched_date,order_delivered_date,order_estimated_delivery_date
0,CA-2014-100006,MH-17785,VEN02,Standard,delivered,2018-05-11 20:07:00.000,2018-05-11 20:30:00.000,2018-05-16 15:14:00.000,2018-05-19 13:42:00.000,2018-05-23
1,CA-2014-100090,PC-18745,VEN01,Standard,delivered,2018-01-30 10:21:00.000,2018-01-30 10:35:00.000,2018-01-31 20:39:00.000,2018-02-14 23:39:00.000,2018-02-26


In [3]:
returns.head(2)

,order_id,return_reason
0,CA-2014-100762,Wrong Delivery
1,CA-2014-100867,Wrong Delivery


In [4]:
products.head(2)

,product_id,product_name,colors,category,sub_category,date_added,manufacturer,sizes,upc,weight,product_photos_qty
0,FUR-BO-10004357,O'Sullivan Living Dimensions 3-Shelf Bookcases,White,Furniture,Bookcases,2016-06-08,Dearfoams,NaN,39161445658,NaN,8
1,FUR-CH-10002044,Office Star - Contemporary Task Swivel chair w...,Blue,Furniture,Chairs,2017-01-09,Easy Spirit,8.5,29021040932,NaN,2


In [6]:
transactions.head(2)

,TransactionID,Order_ID,Product_ID,Sales_Amount,Quantity,Discount,COGS
0,1,CA-2014-145317,FUR-BO-10001798,261.96,2,0.0,41.9136
1,2,CA-2016-118689,FUR-CH-10000454,731.94,3,0.0,219.5820


In [32]:
merged=pd.merge(orders,returns,how='left',on='order_id')

In [16]:
merged.head(2)

,order_id,customer_id,partner_id,ship_mode,order_status,order_purchase_date,order_approved_at,order_dispatched_date,order_delivered_date,order_estimated_delivery_date,...,product_name,colors,category,sub_category,date_added,manufacturer,sizes,upc,weight,product_photos_qty
0,CA-2014-100006,MH-17785,VEN02,Standard,delivered,2018-05-11 20:07:00.000,2018-05-11 20:30:00.000,2018-05-16 15:14:00.000,2018-05-19 13:42:00.000,2018-05-23,...,GBC Recycled Grain Textured Covers,Blue,Office Supplies,Binders,2017-01-23,Callisto,"7,6",8.02E+11,NaN,0
1,CA-2014-100090,PC-18745,VEN01,Standard,delivered,2018-01-30 10:21:00.000,2018-01-30 10:35:00.000,2018-01-31 20:39:00.000,2018-02-14 23:39:00.000,2018-02-26,...,Target Practical Foundations 30 x 60 Training ...,Pink,Furniture,Tables,2016-10-05,Polo Ralph Lauren,NaN,8.90E+11,NaN,2


In [33]:
merged=pd.merge(merged,trans2,left_on='order_id',right_on='Order_ID')

In [34]:
merged = pd.merge(merged, products, how='left', left_on='Product_ID', right_on='product_id')

In [35]:
rep=merged.groupby('category').agg(total_orders=('order_id','nunique'), total_returns=('return_reason','count')).reset_index().sort_values(by='total_orders',ascending=False)

In [36]:
rep['return_rate']=round(rep['total_returns']*100.00/rep['total_orders'],2)

In [37]:
rep.head()

,category,total_orders,total_returns,return_rate
1,Office Supplies,3659,470,12.85
0,Furniture,1710,172,10.06
2,Technology,1500,148,9.87


In [38]:
pivot_df = rep.set_index('category')
pivot_df

,total_orders,total_returns,return_rate
category,,,
Office Supplies,3659,470,12.85
Furniture,1710,172,10.06
Technology,1500,148,9.87


In [39]:
pivot_df_2 = pd.pivot_table(rep,index='category',
values=['total_orders','total_returns','return_rate'],
aggfunc='first'
)
pivot_df_2

,return_rate,total_orders,total_returns
category,,,
Furniture,10.06,1710,172
Office Supplies,12.85,3659,470
Technology,9.87,1500,148


In [40]:
pivot_df_2.reset_index().melt(id_vars='category',var_name='metric',value_name='value')

,category,metric,value
0,Furniture,return_rate,10.06
1,Office Supplies,return_rate,12.85
2,Technology,return_rate,9.87
3,Furniture,total_orders,1710.00
4,Office Supplies,total_orders,3659.00
5,Technology,total_orders,1500.00
6,Furniture,total_returns,172.00
7,Office Supplies,total_returns,470.00
8,Technology,total_returns,148.00
